# 03 CBAM 空间注意力详解

上一讲学习了如何给每个通道计算权重。本讲关注另一个问题：**特征图的哪些位置值得相对保留更多信息？** 我们按输入、池化、拼接、卷积、Sigmoid 和相乘的顺序逐步理解。

## 1. 输入和输出是什么

空间注意力接收的是通道注意力调整后的特征 $F'$：

$$F' \in \mathbb{R}^{B \times C \times H \times W}$$

它为每个样本生成一张空间权重图：

$$M_s(F') \in \mathbb{R}^{B \times 1 \times H \times W}$$

每个空间位置一个权重，同一张权重图应用于该样本的所有通道。乘回输入后，输出 $F''$ 的形状仍为 $B \times C \times H \times W$。不同样本分别计算自己的权重图。

## 2. 第一步：沿通道做两种池化

先看一个样本，假设有两个 $2 \times 2$ 的通道：

$$F'_1=\begin{bmatrix}1&3\\2&4\end{bmatrix},\qquad F'_2=\begin{bmatrix}5&1\\4&2\end{bmatrix}$$

对每个位置，看这个位置上的所有通道值。左上角的平均值是 $(1+5)/2=3$，最大值是 $\max(1,5)=5$。对四个位置都分别这样处理，得到：

$$F_{\mathrm{avg}}^s=\begin{bmatrix}3&2\\3&3\end{bmatrix},\qquad F_{\mathrm{max}}^s=\begin{bmatrix}5&3\\4&4\end{bmatrix}$$

两张图各是 $1 \times 2 \times 2$。平均池化概括该位置的通道响应；最大池化保留该位置的最强通道响应。对于 $C$ 个通道，同样是在每个位置对 $C$ 个数求平均、求最大值。这里压缩的是**通道维度**，保留高和宽。

## 3. 第二步：沿通道拼接

把平均池化图和最大池化图作为两个通道叠起来，不是相加，也不是左右拼成一张宽图：

$$B \times 1 \times H \times W\quad +\quad B \times 1 \times H \times W\quad \longrightarrow\quad B \times 2 \times H \times W$$

这里的 $+$ 仅表示列出两张待拼接的图，不表示数值相加。拼接结果的第一个通道是平均池化图，第二个通道是最大池化图。

## 4. 第三步：两通道卷积生成一张分数图

原始 CBAM 使用一个 $7 \times 7$ 卷积，把拼接后的 2 个输入通道变成 1 个输出通道，通常使用合适的填充以保持高和宽不变。卷积核参数的形状可理解为：

$$1 \times 2 \times 7 \times 7$$

第一个 $1$ 是输出通道数；$2$ 是输入通道数；每个输入通道各有一个 $7 \times 7$ 的卷积核切片。

对输出图的每个位置，先分别用两个切片处理两张输入图对应的局部区域，再把两个结果相加；如果该卷积层有偏置，再加上偏置。把所有位置的结果排列起来，就得到一张分数图。两个切片的参数可不同，并在训练中学习。

注意：**相加的是经过各自卷积核切片处理后的结果，不是直接对两张池化图取平均。**

## 5. 用简化例子看相加

为了便于手算，暂时用只看当前位置的 $1 \times 1$ 卷积演示，不代表原始 CBAM 的 $7 \times 7$ 卷积。假设两个切片的权重分别是 $0.1$、$0.2$，偏置为 $0$。

在左上角，平均池化值为 $3$，最大池化值为 $5$：

$$\text{左上角分数}=3\times0.1+5\times0.2=1.3$$

右上角：

$$\text{右上角分数}=2\times0.1+3\times0.2=0.8$$

两个结果是**加和**，不再除以 2。真正的 $7 \times 7$ 卷积还会使用局部邻域内的值，并使用对应位置的可学习参数。

## 6. 第四步：Sigmoid 得到空间权重图

对分数图的每个位置分别应用 Sigmoid，得到 0 到 1 之间的权重。以上面的简化例子为例：

$$\sigma(1.3)\approx0.79,\qquad \sigma(0.8)\approx0.69$$

它们分别是左上角和右上角的权重。所有位置都计算完成后，得到 $B \times 1 \times H \times W$ 的空间权重图。

把整个过程写成公式：

$$M_s(F')=\sigma\left(f^{7\times7}\left([\operatorname{AvgPool}_c(F');\operatorname{MaxPool}_c(F')]\right)\right)$$

其中下标 $c$ 提醒我们沿通道池化；方括号表示通道拼接；$f^{7\times7}$ 表示卷积。

## 7. 第五步：乘回每个通道

空间权重在通道维度广播，同一位置的权重应用到所有通道的该位置：

$$F''=M_s(F')\odot F'$$

例如左上角权重约为 $0.79$，输入两个通道的左上角分别为 $1$ 和 $5$，则输出分别约为 $0.79$ 和 $3.95$。其他位置也对应相乘。输入、输出形状相同，数值通常改变。

这张空间权重图不是每个通道各算一张，而是**每个样本算一张、该样本的各通道共享**。空间权重越大，当前位置受到的抑制越少；它不保证一定准确标出物体，或单独解释最终预测。

## 8. 容易混淆的地方

1. **平均发生在哪里？** 在池化阶段：同一个位置，对原始 $C$ 个通道的数求平均。
2. **最大值发生在哪里？** 同样在池化阶段：同一个位置，取 $C$ 个通道中的最大值。
3. **两张池化图直接相加或取平均吗？** 不。先沿通道拼接，再分别与卷积核切片计算；卷积结果相加，不再取平均。
4. **64 个输入通道会直接送入这一步卷积吗？** CBAM 空间注意力先把它们池化成 2 个通道，因此卷积接收 2 个通道。
5. **卷积参数怎样更新？** 空间权重影响后续网络的预测；训练时，损失经反向传播产生梯度，优化器与其他参数一起更新卷积参数。

## 9. 小结与自测

完整形状变化：

$$B\times C\times H\times W\ \rightarrow\ \text{两张 }B\times1\times H\times W\ \rightarrow\ B\times2\times H\times W\ \rightarrow\ B\times1\times H\times W$$

最后的空间权重图乘回输入，得到 $B \times C \times H \times W$ 的输出。

1. 空间注意力的输入是原始特征，还是通道注意力的输出？
2. 输入 64 个通道时，一张池化图上某位置的值是如何得到的？
3. 拼接后的两张图是否直接取平均？卷积时分别使用什么参数？
4. 空间权重图怎样应用到 64 个通道？不同样本共享同一张权重图吗？
5. 原始 CBAM 的 $7 \times 7$ 卷积相比示例中的 $1 \times 1$ 卷积，还能利用什么信息？